# 1. Scrappe games

In [2]:
!pip install gradio requests pandas bs4 

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/57.8 MB ? eta -:--:--
   --------- ------------------------------ 13.1/57.8 MB 65.7 MB/s eta 0:00:01
   ----------------------- ---------------- 34.3/57.8 MB 82.9 MB/s eta 0:00:01
   ---------------------------------- ----- 49.5/57.8 MB 79.0 MB/s eta 0:00:01
   ---------------------------------------  57.7/57.8 MB 78.5 MB/s eta 0:00:01
   ---------------------------------------- 57.8/57.8 MB 66.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ---------------------------------------- 11.5/11.5 MB 66.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 71.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 44.1 MB/s e

In [3]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import gradio as gr
import time
import warnings

c:\Users\r107023\Anaconda3\envs\pyt_310\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:


def scrape_nba_games(days=10):
    today = datetime.today()- timedelta(days=1) #por si quiero buscar partidos de meses anteriores
    collected_games = []
    
    for i in range(days):
        date = today - timedelta(days=i)
        url = f"https://www.basketball-reference.com/boxscores/?month={date.month}&day={date.day}&year={date.year}"
        print(url)
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        
   # For each div with class="game_summary"
        for div_game in soup.select("div.game_summary"):
            table_teams = div_game.select_one("table.teams")
            if not table_teams:
                continue

            rows = table_teams.select("tr")
            if len(rows) != 2:
                continue

            team1_anchor = rows[0].select_one("a[href^='/teams/']")
            team2_anchor = rows[1].select_one("a[href^='/teams/']")
            score1 = rows[0].select_one("td.right").text.strip()
            score2 = rows[1].select_one("td.right").text.strip()

            # Get boxscore link from either row's 'gamelink'
            boxscore_a = table_teams.select_one("td.gamelink a[href^='/boxscores/']")
            if not boxscore_a:
                continue
            gameid = boxscore_a["href"].split("/")[-1].replace(".html", "")

            if team1_anchor and team2_anchor:
                team1_id = team1_anchor["href"].split("/")[2]
                team2_id = team2_anchor["href"].split("/")[2]
                collected_games.append({
                    "date": date.strftime("%Y-%m-%d"),
                    "team1": team1_id,
                    "team2": team2_id,
                    "score1": score1,
                    "score2": score2,
                    "link": boxscore_a["href"],
                    "gameid": gameid
                })

    # Load or create CSV, then merge without duplicates
    df_new = pd.DataFrame(collected_games)
  
    if os.path.exists("games.csv"):
        df_old = pd.read_csv("games.csv", dtype=str)
        df = pd.concat([df_old, df_new]).drop_duplicates("gameid").reset_index(drop=True)
    else:
        df = df_new.drop_duplicates("gameid").reset_index(drop=True)
    df.to_csv("games.csv", index=False)
    time.sleep(2)
    return df


df_games = scrape_nba_games(20)



https://www.basketball-reference.com/boxscores/?month=2&day=8&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=7&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=6&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=5&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=4&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=3&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=2&year=2025
https://www.basketball-reference.com/boxscores/?month=2&day=1&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day=31&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day=30&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day=29&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day=28&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day=27&year=2025
https://www.basketball-reference.com/boxscores/?month=1&day

In [13]:
df_games.sort_values(by='date', ascending=False)


,date,team1,team2,score1,score2,link,gameid
338,2025-02-08,GSW,CHI,132,111,/boxscores/202502080CHI.html,202502080CHI
339,2025-02-08,HOU,DAL,105,116,/boxscores/202502080DAL.html,202502080DAL
340,2025-02-08,UTA,LAC,110,130,/boxscores/202502080LAC.html,202502080LAC
341,2025-02-08,IND,LAL,117,124,/boxscores/202502080LAL.html,202502080LAL
342,2025-02-08,OKC,MEM,125,112,/boxscores/202502080MEM.html,202502080MEM
...,...,...,...,...,...,...,...
333,2024-12-16,PHI,CHO,121,108,/boxscores/202412160CHO.html,202412160CHO
334,2024-12-16,MIA,DET,124,125,/boxscores/202412160DET.html,202412160DET
335,2024-12-16,UTA,LAC,107,144,/boxscores/202412160LAC.html,202412160LAC
336,2024-12-16,DEN,SAC,130,129,/boxscores/202412160SAC.html,202412160SAC


# 2. Scrappe box score

In [ ]:
def scrape_box_scores():
    # 1. Load CSVs if they exist, otherwise create empty DataFrames
    if os.path.exists("games.csv"):
        df_games = pd.read_csv("games.csv", dtype=str)
        
    else:
        df_games = pd.DataFrame(columns=["date", "team1", "team2", "score1", "score2", "link", "gameid"])
        df_games = df_games.sort_values(by='date', ascending=False)
    
    if os.path.exists("box_score.csv"):
        df_boxscore = pd.read_csv("box_score.csv", dtype=str)
    else:
        print("No box_score.csv found")
        df_boxscore = pd.DataFrame()

    if os.path.exists("box_advance.csv"):
        df_boxadvance = pd.read_csv("box_advance.csv", dtype=str)
    else:
        print("No box_advance.csv found")
        df_boxadvance = pd.DataFrame()

    # 2. Iterate over games, skipping those already in box_score and box_advance
    existing_boxscore_gameids = set(df_boxscore["gameid"]) if not df_boxscore.empty else set()
    existing_boxadvance_gameids = set(df_boxadvance["gameid"]) if not df_boxadvance.empty else set()

    new_boxscore_rows = []
    new_boxadvance_rows = []

    for _, game_row in df_games.iterrows():
        gameid = game_row["gameid"]
        # If gameid already exists in both, skip
        if gameid in existing_boxscore_gameids and gameid in existing_boxadvance_gameids:
            #print("skipping", gameid)
            continue

        url = f"https://www.basketball-reference.com/boxscores/{gameid}.html"
        print("Fetching:", url)
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")

        # Check for 429 or similar limited response
        if "Limited Request (429 error)" in soup.text or "Too Many Requests" in soup.text:
            print("Encountered rate limit on:", url)
            time.sleep(10)
            continue


        print("Parsing:", gameid)

        # 4. Basic Box Score Stats
        basic_tables = soup.find_all("table", {"id": lambda x: x and x.endswith("-game-basic")})
        for tbl in basic_tables:
            # Extract teamId from table id: e.g. "box-DEN-game-basic" -> "DEN"
            team_id = tbl["id"].split("-")[1].upper()

            # Each player row in <tbody>, ignoring <thead> or <tfoot>
            for row in tbl.select("tbody tr"):
                if "thead" in row.get("class", []):
                    continue
                # Identify if there's a link to a player's page
                player_cell = row.find("th", {"data-stat": "player"})
                if not player_cell:
                    print("No player cell found")
                    continue
                player_link = player_cell.find("a")
                if not player_link:
                    print("No player link found")
                    continue
                # Player ID from the link (e.g. "/players/j/jamesle01.html" -> "jamesle01")
                player_id = player_link["href"].split("/")[-1].replace(".html", "")

                # Gather stats from data-stat attributes
                stats_dict = {
                    "gameid": gameid,
                    "teamid": team_id,
                    "playerid": player_id,
                }
                # Loop each <td> with data-stat
                for stat_cell in row.find_all(["td"]):
                    stat_name = stat_cell.get("data-stat", "")
                    stats_dict[stat_name] = stat_cell.text.strip()

                print(stats_dict)
                new_boxscore_rows.append(stats_dict)

        # 5. Advanced Box Score Stats
        adv_tables = soup.find_all("table", {"id": lambda x: x and x.endswith("-game-advanced")})
        for tbl in adv_tables:
            team_id = tbl["id"].split("-")[1].upper()

            for row in tbl.select("tbody tr"):
                if "thead" in row.get("class", []):
                    continue
                player_cell = row.find("th", {"data-stat": "player"})
                if not player_cell:
                    print("No player cell found")
                    continue
                player_link = player_cell.find("a")
                if not player_link:
                    print("No player link found")
                    continue
                player_id = player_link["href"].split("/")[-1].replace(".html", "")

                stats_dict = {
                    "gameid": gameid,
                    "teamid": team_id,
                    "playerid": player_id,
                }
                for stat_cell in row.find_all(["td"]):
                    stat_name = stat_cell.get("data-stat", "")
                    stats_dict[stat_name] = stat_cell.text.strip()
                
                new_boxadvance_rows.append(stats_dict)

        time.sleep(1)  # Prevent overwhelming the server

    # 6. Append results to box_score.csv, box_advance.csv with no duplicates
    if new_boxscore_rows:
        new_boxscore_df = pd.DataFrame(new_boxscore_rows)
        df_boxscore = pd.concat([df_boxscore, new_boxscore_df], ignore_index=True).drop_duplicates(
            subset=["gameid", "teamid", "playerid"], keep="first"
        )
        df_boxscore.to_csv("box_score.csv", index=False)
    else:
        print("No box stats found:",gameid )


    if new_boxadvance_rows:
        new_boxadvance_df = pd.DataFrame(new_boxadvance_rows)
        df_boxadvance = pd.concat([df_boxadvance, new_boxadvance_df], ignore_index=True).drop_duplicates(
            subset=["gameid", "teamid", "playerid"], keep="first"
        )
        df_boxadvance.to_csv("box_advance.csv", index=False)
    else:
        print("No advance stats found:",gameid )

    return df_boxscore, df_boxadvance

df_boxscore, df_boxadvance = scrape_box_scores()

Fetching: https://www.basketball-reference.com/boxscores/202501130NYK.html
Parsing: 202501130NYK
{'gameid': '202501130NYK', 'teamid': 'DET', 'playerid': 'harrito02', 'mp': '36:06', 'fg': '3', 'fga': '12', 'fg_pct': '.250', 'fg3': '1', 'fg3a': '3', 'fg3_pct': '.333', 'ft': '4', 'fta': '4', 'ft_pct': '1.000', 'orb': '2', 'drb': '6', 'trb': '8', 'ast': '3', 'stl': '3', 'blk': '0', 'tov': '0', 'pf': '1', 'pts': '11', 'game_score': '11.7', 'plus_minus': '+12'}
{'gameid': '202501130NYK', 'teamid': 'DET', 'playerid': 'hardati02', 'mp': '35:22', 'fg': '4', 'fga': '12', 'fg_pct': '.333', 'fg3': '2', 'fg3a': '8', 'fg3_pct': '.250', 'ft': '0', 'fta': '0', 'ft_pct': '', 'orb': '0', 'drb': '3', 'trb': '3', 'ast': '3', 'stl': '0', 'blk': '0', 'tov': '0', 'pf': '0', 'pts': '10', 'game_score': '6.2', 'plus_minus': '+5'}
{'gameid': '202501130NYK', 'teamid': 'DET', 'playerid': 'cunnica01', 'mp': '32:03', 'fg': '14', 'fga': '27', 'fg_pct': '.519', 'fg3': '4', 'fg3a': '8', 'fg3_pct': '.500', 'ft': '4', 'f

# 3. Explore the games

In [11]:
import os
import pandas as pd
import gradio as gr


def load_and_filter_games(date_filter, team_filter):
    if not os.path.exists("games.csv"):
        return pd.DataFrame(columns=["date","team1","team2","score1","score2","link","gameid"])
    df = pd.read_csv("games.csv", dtype=str)

    if date_filter and date_filter != "All":
        df = df[df["date"] == date_filter]

    if team_filter and team_filter != "All":
        df = df[(df["team1"] == team_filter) | (df["team2"] == team_filter)]

    return df

def on_table_select(gameid, team1_id, team2_id):

    print("gameid", gameid, "team1_id", team1_id, "team2_id", team2_id)
    if not os.path.exists("box_score.csv"):
        return pd.DataFrame(), pd.DataFrame()

    box_df = pd.read_csv("box_score.csv", dtype=str)     
    team1_stats = box_df[(box_df["gameid"] == gameid) & (box_df["teamid"] == team1_id)]
    team2_stats = box_df[(box_df["gameid"] == gameid) & (box_df["teamid"] == team2_id)]

    return team1_stats, team2_stats

def handle_selection(evt: gr.SelectData, df):            
    #if evt.index[1]==6: #solo funciona para la columna gameid        
    gameid = evt.row_value[6]
    team1_id = evt.row_value[1]
    team2_id = evt.row_value[2]
    return on_table_select(gameid, team1_id, team2_id)

def update_table(date_filter, team_filter):
    return load_and_filter_games(date_filter, team_filter)

def launch_interface():
    global global_df
    global_df = load_and_filter_games(None, None)

 # Custom CSS for the page
    custom_css = """
    body {
        background-color: #ffffff;
        font-family: Arial, sans-serif;
    }
    .gr-block {
        border-radius: 8px;
        margin: 10px;
        padding: 15px;
        background-color: #f5f5f5;
        box-shadow: 0 2px 6px rgba(0,0,0,0.15);
    }
    .gr-rows, .gr-dataframe {
        border: 1px solid #ddd !important;
        font-size: 0.9em;
    }
    .gr-dataframe th, .gr-dataframe td {
        padding: 8px;
        text-align: left;
        border-bottom: 1px solid #ddd;
    }
    .gr-dataframe th {
        background-color: #f2f2f2;
        font-weight: bold;
    }
    .gr-dataframe tr:hover {
        background-color: #f1f1f1;
    }
    """

    with gr.Blocks(css=custom_css) as demo:
        gr.Markdown("<h1 style='text-align:center; color:#323232;'>NBA Games Dashboard</h1>")
        gr.Markdown("<p style='text-align:center;'>Browse or filter games, then click a game row to see each team's stats.</p>")

        with gr.Row():
            date_choices = ["All"] + sorted(global_df["date"].unique()) if not global_df.empty else ["All"]
            team_choices = ["All"] + sorted(set(global_df["team1"]).union(global_df["team2"])) if not global_df.empty else ["All"]

            with gr.Column():
                date_dropdown = gr.Dropdown(
                    choices=date_choices,
                    value="All",
                    label="Filter by Date"
                )
            with gr.Column():
                team_dropdown = gr.Dropdown(
                    choices=team_choices,
                    value="All",
                    label="Filter by Team"
                )

        games_table = gr.DataFrame(
            headers=["date","team1","team2","score1","score2","link","gameid"],            
            value=global_df
        )

        gr.Markdown("<hr/>", visible=True)
        gr.Markdown("<h2 style='color:#323232;'>Team Stats</h2>")

        with gr.Row():
            stats_table1 = gr.DataFrame(label="Team 1 Stats")
            stats_table2 = gr.DataFrame(label="Team 2 Stats")

        # Dropdown changes -> update table
        date_dropdown.change(
            fn=update_table,
            inputs=[date_dropdown, team_dropdown],
            outputs=games_table
        )
        team_dropdown.change(
            fn=update_table,
            inputs=[date_dropdown, team_dropdown],
            outputs=games_table
        )

        # Load initial data
       # games_table.update(value=global_df)

        # On table row click -> load team stats
        games_table.select(
            fn=handle_selection,
            outputs=[stats_table1, stats_table2]
        )

    demo.launch()

if __name__ == "__main__":
    launch_interface()

c:\Users\r107023\Anaconda3\envs\pyt_310\Lib\site-packages\gradio\utils.py:1017: UserWarning: Expected 1 arguments for function <function handle_selection at 0x000002DB84F56AC0>, received 0.
  warnings.warn(
c:\Users\r107023\Anaconda3\envs\pyt_310\Lib\site-packages\gradio\utils.py:1021: UserWarning: Expected at least 1 arguments for function <function handle_selection at 0x000002DB84F56AC0>, received 0.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


c:\Users\r107023\Anaconda3\envs\pyt_310\Lib\site-packages\gradio\helpers.py:968: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")


gameid 202502010CHO team1_id DEN team2_id CHO


In [7]:
pip show gradio

Name: gradio
Version: 5.15.0
Summary: Python library for easily interacting with trained machine learning models
Home-page: https://github.com/gradio-app/gradio
Author: 
Author-email: Abubakar Abid <gradio-team@huggingface.co>, Ali Abid <gradio-team@huggingface.co>, Ali Abdalla <gradio-team@huggingface.co>, Dawood Khan <gradio-team@huggingface.co>, Ahsen Khaliq <gradio-team@huggingface.co>, Pete Allen <gradio-team@huggingface.co>, Ömer Faruk Özdemir <gradio-team@huggingface.co>, Freddy A Boulton <gradio-team@huggingface.co>, Hannah Blair <gradio-team@huggingface.co>
License: 
Location: c:\Users\r107023\Anaconda3\envs\pyt_310\Lib\site-packages
Requires: aiofiles, anyio, audioop-lts, fastapi, ffmpy, gradio-client, httpx, huggingface-hub, jinja2, markupsafe, numpy, orjson, packaging, pandas, pillow, pydantic, pydub, python-multipart, pyyaml, ruff, safehttpx, semantic-version, starlette, tomlkit, typer, typing-extensions, uvicorn
Required-by: 
Note: you may need to restart the kernel to us